# A LoRA linear layer

Wrap a pretrained linear layer with a low-rank adapter whose base weights never change.

```python
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        # Your code
    def forward(self, x):
        # Your code
    def merge(self):
        # Your code: return a new nn.Linear
```

- `base` is an existing `nn.Linear(d_in, d_out)`, with or without a bias. Store it as `self.base` (the same object, not a copy) and freeze **all** its parameters (`requires_grad=False`). Do not change its values.
- Create `self.A = nn.Parameter(torch.randn(r, d_in) / math.sqrt(d_in))` and `self.B = nn.Parameter(torch.zeros(d_out, r))`, both trainable. These two are the only trainable parameters.
- `forward(x)` takes `x` of shape `(..., d_in)` and returns `base(x) + (alpha / r) * x A^T B^T`, shape `(..., d_out)`. Multiply by `A` first.
- `merge()` returns a **new** `nn.Linear(d_in, d_out)` (with a bias exactly when `base` has one) whose weight is $W + \frac{\alpha}{r} B A$ and whose bias equals `base`'s, so that `merge()(x)` equals `self(x)`. It must not change `self`.

**Example.**

```python
torch.manual_seed(0)
base = nn.Linear(6, 4)
lora = LoRALinear(base, r=2, alpha=4.0)
x = torch.randn(3, 6)
torch.equal(lora(x), base(x))                                       # True: B starts at zero
sum(p.numel() for p in lora.parameters() if p.requires_grad)        # 20 = 2 * (6 + 4)
[n for n, p in lora.named_parameters() if p.requires_grad]          # ['A', 'B']
```

After any number of optimizer steps on `lora.parameters()`, `base.weight` and `base.bias` must be exactly unchanged.

**Restrictions.** Only `import torch`, `import torch.nn as nn` and `import math`.

**Constraints.** $d_{\text{in}}, d_{\text{out}} \le 4096$, $r \le 64$. Time limit 1 second per call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem4_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem4_submission.py
import math
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return x

    def merge(self):
        # Your code
        return self.base


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem4_submission as submission
importlib.reload(submission)
# e.g. submission.LoRALinear(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XAtz2za2/iu46syazMqM7DhpVql7107dbnbymiQ73TuWh4VESGJNkSxBRXa9/u/3fAcACerh2GnvTXdkCQQOzvsFcG96adIbit7y4Nl+OTjq9UVvoWpJQzebT+q0zhTGTsTr4sOJyNJcyUpk8lpVeJ4prYvcLcoGTxlcmi9rpWn04Cn9TNLpNJ0ss/qaRp7QgMrrCt97APmaIfKyIllmKs7lgncsq2KcqcVRrJfjRap1SvvQLJllxUolcbooi6rGJue9uqgmcwYh63nvgr6MZZ4rEHNuf8VZUZSYPZWZVqBM6Tqu04UqlnWMB4cDjMqZAZkVlWQGAHQ0rZT6Pc1n7YhBFr+nREC9zPEUe+liWU2YgKJKZ2kuM0wqZVUf0mBdLdUtL8qYQTe9uaxyYmJUMkdG/N8/VFaqSot6Lmth+SBmlUwwaAgXwS/TqlgIu9yNPvoljEb5KD+x00WqxaTIa2L5Y8Kx1o+/w5/vH1uo9LtMk+8fm9mERCRe1SJRIIr2Jx6J6TKf1MR8LSCZhAfjRwa3Wl4q+qZGeSskYVgjfqGhXyJx9u+TN+9fn30k1dE1SFIGLIBpoefFKhe06MMyF8HnVKeEVfhCfAS4WlRL2lZ9VtU1LyLSfpRptqxoZSVTrQR+ilVaz4UUBE/LmRKrKq2JYDEtKnGq8qHQ8lqsgO5Kanpa5LO+yAFVJMtFKeZpktD0NC+XtQb3jBRyy9JJUV43P6Aw4K/9mdP6a0FQ8xKj34iXRSbH4uX7f2khKyU0dLUCr3KgsqfFm0Pxvir6QhciKyYyA0RSHS0WS2LKWIGpxKICJpPS4ywFH8ZE/5xEOZUTUr8IOx1GAwhXitkS8g/MzppgZWq/nldKJkKXigRGs6piOZtn16Qt2VQQLoTFpKjA6EqB6bVIp0CQfrJVWTqseYApH9+fnf0Q/3jy8tO7D+IYu4PgUT7JJO0OMQRnVxNVQlPC4SgX9K+kR2YWKRRxXCUB6WLSFws9c3No37wAk/PEjuBfK9wAc1sgcV4GV24teRFvjZMQzNMbnRL9aa5rmU9UcNU3z6NPKtdFFXrLeVsy5CoXV1FCDnEyD8JoUi7pk8UchGayYjLFK97urKqKyoNiSPZg5WUktawqeU14t3TouSxVMCvqPilljk/Sz+NRjzSbHOWo5zDTxOt6WWYqAOU0P4x4qcWFeUpz3CTACvtiOurdAOAtCVybvcSNJr+jrko1qUknbrz5t7Rdi9kkK/RdmPVFVRfZ8YHaf9YX0n597vAly1oRyg2yUtfXROg0KyQQwzjv2X3gETMz9IGklfm6gxxDgJ0ebqPNPWLymg1IIKkm95bWtJrwyLIgFOQpoIX+s5V95m8PTypT9oQ51qT5tAFuFRnyzjLLRGKGZRc+LLvw4SueqiriGDMCyC3kFf7IsQ5mYl+swjAEbCI0/V0JRcFLDGB8W0ylyyd2dOKzzMg/iCCT1QxuV0FhxQ39GUZPZrdhR/bqt6XM7qGVXaMyU4F0nrCm90PmKCEh67qyz0c9FkcLZIu6eArSzvk/VwtfNRj/2PKBkNswJp+rDQRIZbiOMSkDsGXiWxjkjG/o0X9VHdQwi4Y6wkh1zCbC4sA3vVMevKEnkXbJBv4YJVZYwGEUc74Vxx10ZuT+IDm7656btOegdjGNdS5LiuT1/69rzopcPdAp05ZOK6+gkhzbeyGFxIStt4tPkKQTYiQSl9DHp0tcBzsADBKlyuNPlOl5CmYR/ETsW0dvE0DYCSEYigCTn111GE9iCWRfjL+e7/JefDdTjF3wfl/Ldgm2MxzddQV2I2k20cE43OZraHnrZ7Yub/zE2H5l2XYtGxT0jbcjzc59WTk4DMFLXop4sSSjoOwmmOZ98YjcKVnGo0eXK3xzmJjU8SVFADHNeecpElTQQDkOZYEzMi+ZU8pYzZYLSsophIgykxMVuayToz7sifzieWtZMuR8VmIBdrwwMyk7Q/DIg3WEzGMsSUmRiWAdYqmiVEZVkuLb72kZmDW8R0fB2Z80ymWdyDUVNqSLiUKCn05TZJZImVtavrlJxV/Fwe0LsSqqS2T1kpVXQHyUjzb+0jKZkG85jCQ+zQOtkAoSVvfl8gufxeRd/ZR7SybN5SPFQ025N1EABnGuu9espOTd1FDY6h2BqFCCUYG00CCJ8/A3ciImtAGWEJEm60a1wok9kgOqLvbHy4Q8KYY5B6e0PEUBtSzJnES9SicKAIlDKUXoYmotmHB2HNWsQhiZUr6E8A0ItGJJzyLxCQ/SioaJmGwPWyAumM3AYZV01KoeIJskCBFVl9OYwajKeZsvqFJSb19MOUpt0xFL77GwQhSPhV82tCoZQxUrWENw6OsdCZG2+e7YQlpzRWOqaS7bIXnYF5eHtFvXPzLK/bXBDiVfZgXjCUYcMhsOvWFmA1VsQYIQu4Mdfr7pE8R2VBcF6w0V/0k9jA6nt5Tlk5apF7bqapSYlYnMjgGJZY56/sayxq7jGpIKOyfiLf94S6osqQyekmrNkMCaIvPGyokBwTruNNAqn5F1qsQJrC1wSJJJsSBuTyWpX9xMHHHDZqmlTIuYOwydLsdPGCELIMza5kHKTmOeZgmsboLOCBXMKWwBM//58d1bbkLBcnSdEIZsq+V1PaeR/VfC7SO+sy2OOEmr78V37R70kH6TVf6Hx+rvN0p+Mjav/CfBXtVZOm6GzB8aiZZ1mrXDRfP1V120TYJS1nN/uU5nuWyX6Wu91l5wPyqKC2M5uWxGSIvRZ9Km16AnKZGZq8bd0cBlWu9niqaJg+hZdGBzOj1En2iBZl2fXQpVFzb4OOdXypS4PBvlbo9ommak0e5nMOoR2kWlUAHaTgtlotGjf+WXOXo4usjQUSm4A0AbpiWkFj2ymeI/zj6ckeVYXkTv6W8Qx2iDxXEYkePCcsrqSlkRZqOcuBJhckTaqqo6GJAxUPoAMCEB3Nb74l6Q+IZi9W9yKM6OBofcCXv988n/fIxPT96+PfuBMLgZ9WLbPIxjEEOEOXman+pKTew3yvPNt0mxKAnXUe92lL9+9+59/PbdD2cfCVxAuhL9WJCt4svPpLjKfD3R1/mkefCacsiXBMP8+qi8Hz9Qktn++knliM9FdXZVhn6L5ZPpxXypyxJLqvQWAXRsuSB/Q2FLNRbLpaID5HcjkNlM4slcTS51oKsJRFzLbrz9YOwd2TAFK/CxuDTMWegZMeYFuUVkgKbHx2aj671GOpwM2VYiAlVdpaat2AlRJgidX2zLZutKUWLHXCItofqaEO3moR+vqT6/4jwUTTm1mR+eW7SHpC3cB7bYD+EmNS+3JXKRGz9zoyL8zYvbIb7TbCp+bi2CthmNkED8isgxk52sdajBo/MLi+hYNuMcJ2sXcrgdGi81A/PGx2QPsemV2nGoOKWMZLtpQqaCTKwyiYIBTBDM/AAVDZLudIx2PJFzybmbyiIott2A++IezxeyXiwzf2TK3RFy1fDNxPyVzC4DCGMtdHtpOqYbfTalwXpV0aSymBgxusPNAGZbKpQvB5InISlfoLVgfka0GzESmWOwoMyT3BGKOQBfMLJGEOFwe2z0ZBHJJLF7hJuTW9l48yJdZinEjT3P9w8utixshRctywRZ95a1fpGY3c3GH8nrrVNDAiVZMR9t453Ibw1qCy95hWUkff8DXFznIEFb44LHug4HtmnA2tJdzPuqpUznQ5l+4mxnnfY1hWA8YGj3gvrW88h3AUyT3VRtn7RtxzZYre/Jhh/JslR5EriOj1k06hmXZ9z7f3f5tW0XjoKnaf7ORrGT5exEIwaFpreytiAq7Lw3sn5DKeOGThsf9EDkCDFPIT2QSBItqJ3u32UBxkqMymOUDQLzol8LlKfsYANvnzDc7l3+iohCjHGgbvb6Ys8AcRYFA9yj/ajune2hFXu7vZ+4DX+0LLagrx3uLbBvmhPEscw1eDuZ2wjBqWAbILj9rCQVsf/mv+ELU41mK3mt982RKlUylBemDSAopYVW5Nl13+1JlUsbagKjBaaJZFMpJB3XIlMzKs1XSJsEz4yiKPSCGSVlIikgFAN4nnJ9aaQQeEHzLxwcvShs0LXBl9TwPyLwYulfRCcv9FSIdniw6hAGWlj+AH6gwy26Q5DDPyLhvHB7NEeyrZQJ8y7xsTGiuChNPtnttmG6MbI107sPwagoSS/+LhxslKye3YjAZE52h/PBha/cW8h/CAvaXe8inp3bBsk8+nUEmzSJlb1DKA3DjKHUlZqrnIvYJnfkVX8u/Zay2zsq9WmVEpjsOuZMNiAQVOjGqKa6Gf27XO0znonSkyrlqkKUpKpk3zNhT6J5QuqOr1GwkbkqzNN8WN/tMI3RVnF1a0R1M32v43ocqCiOmwdU8tlkE8Ap05xyUJ9iHwJBkpqi/FQu50Pl15Bgs9IVWiZoWMHLcm/SKh19Ih3zsvbOEETBCoPZfJzWom+ZiRMTPiFR3vkIsv/bG971FvMdsxeUiQdNLUaedr3KRSUrq9nn8yZFdKTsnHl44ZXEllNIwI9FM+WJS9dJ72kcDYcoK2Sig4CReAy1gU1wK6IHeBSz0MkInLOj8gkQLTL+BMcN7iYei027MAXjyN6vGfFlGt3+NI0Zo7K2koYmYuBtQSLw4Z+30C6Azo5StDF0ZLAwQX1usLrgaIFfbCUXfjrrSjqXz2ruO25s63sJ7loEzE5cDtGBme5HeaMlruU5JaTTIvpYo33y6p1jnmnyROZPYH99fPXTyesPb/q2SrdTyZ3F3Nx1p8CeL+veUgIvDwcOl251zJdg2mYViTNJK0XGZ4QREKbraZbDUdUptqgcmq8+vXpz9iH+cHbyut9gt5bm6FJBe7qtsAijMXozprOD7iQfmXgUebe8jGq0jTkSk+eq1vZbjje3s7B4Q2wd4GN9IVmMmafP8ZgLhguj+JsksQ2pKkIPKDbL4HnCr2DcoNubsG2XTkvCKqI1jgvjzQyN8MDs1djj1kVxyWcK4sbJYxgN0AF2rsvuciq1ajpEpgfStsROX58NBgctAg8iYxfG9mCI0cRJDG6IFVZQNgm6Ix41Zr0BehujnF/BvqTP0Cg+iQ+oAD8aDAbDi682ZL6T2O3j8JB/v8PMcWfF7kSTaOmbR+GXhdvyKimUSZrMZTuKLgBx69ftDyRh9jCbHPXMpT9YYRMymnuADd32HuGXbW/mGd9sly0ZaHaWve5nG1uuyjNT4Bnc1UGXxFu3h1iDpCGgQmVqHDu+wbd/lpV2e0RprRY64NszeTcqsFd15/44QJGERjAN/YYXeQkcQnJKgh3XkmcOx9D+akl+yygFshW+RZA70tYcLtxzmi+VL7+JCbDGHw4ZyPnT4QX5bz4XJmxtl1qwexz1LGg32dsNT9ENRtmzrUyx/tYc+mBoEA1uH3Imtnnx4Suizh+NPPZwbtMrf5VLs0I4bzkHS0XivXl3Y8ODt4tN7mFsnJ11VuQzdy6z7rNFkObmlhln830UEFNZCXcyGPpOwG7OhxtrPe0/SHCDM9LrLXdVHhRL/ixc/HsGfF6R3DOGtLAaBQe8irQ4CXYd1PbFk23hzaazF64gqxrH9uBQ1CwwueoxSsegzV3vykrZqYyLIgs2EDPPAKruaC7A1R1wzQpXoOwMKXyteSpcwWO8WxyjvsFB2dAdD6DcMSe7TbDAmW73rpH/K8pZe/J897Fdn4+4++byaV/YC37epZu+vR8CJJsb5RQDrC9XV3JRwide+IXZpTLBOYnTHJ8ksL7AgVxWznEFKJX62JTYOK4+HjSXqMwNf5njfpDmK7bmONt0YTUiVp5H5s2FoAOdYeLDzl7Bp2JJtFLpbF7vuL42bqZh7cYkbm2msEDUqyih3NlNJU0mG7WvUgSA09IZbr8iiKV9UPGGAzNHlhaGu42OqyzNnO49WheuDSB0XTQn9MAuREPIoAFzzKYRc80BRVuBB/g1Er7hsEIxP0s/q1wEeIy7R6IY/0qhJOzu699E8xhLCjIwVhGM+WI88IBj3ZgPRpKgqIziyrWDy552F2bdZS3udXiMwSsE3AJtsGqzBT0vYXvnFLNPwIqgMrqHnWjslMecGoahn+KWJMYuRw1I5ubaxSxPiiWL8L3EKS95NXNRixl+g/W3a3JsJm7efTWXY0t7OZabLvNyA9zGRdty20VbWnm7fpXEEGV0YjUA/73jZ1XHcmzp9q+YzJCQsPyaY3Eyih2GyamIdTjmukngpx6AHp3w/crYKhHureSBfWB9zsxtdDzz82yedLpr9emO1U1DrqgWy0wGPhOuGj/U3Ln8eZevoFCDBDWkOBjwIsrVq1A8cmhtTvy7o3fjkX/OfuUv+Dn6tN5SpP22OKVmDR+42MeuBcxmx26q05L0vTSnbsPWgzmOxPQ/VGCN337WF0d9cUifkcsarhqNMOx/wrNatt+tLQdhh3z/UqdB5Mp1b/j1DDZUbHEUmnshjP4Qq8tl6wDMZX/j/omQ4Ko7G4ke7h+aVSI4pZ0HxjDh8FpvaCaE8BLuvYpn/u2A9uAjd0UP+xuWNL+bRfmQNXBb95SUkP+2pIxcs0F03oMwIDmGnpjKgBzUBZt8g3tNKViO4ki0kLkJzv6g80bJ+d4JTrhO9y5EwK/LKfviFtPWLB+KNYwa586BbEEeBVdUcWQEGlsCH0IayMLrfD4pN/mtRw7tMSZQHv6Hg86ddlZYK7r0d6MjG3obb1FaSv4oq8TH8744eBYNbHZx6GPIWZ31HUB2YF82Yc3p7kqptlEWrmDRlf9dVUXDNo3T/3Vbp+Q0MPVv4O+Kd0po/j5wFI8eif1B9DQU31EZeEC+xBtkzq2jcWK8PwWeRFaJSNTnlJ+Qv0+G0dHUVwY5hoXdeCB5xiaHScYrghabqHtvz3BAXP22j5dJWwa7TL4TTY52+A1yKk8NmPv6jqfh/eKMcS3YPdjmTczG37KgLfE4LmoQCu9yLzviCPPgSQekQRLqTtI44bzotIXcvsN11Pqap+uywXluLBNZ1ihfAWungExE74qIKPobO/DnjYiebRPRt3bwhGadDrZos02C+xvRrpNCF2XdyBiXEBfRSSIXPwcbzoPgVKCX8DNxNk7URF4fkxXsUJaDZ4ac+6rK8zDcetf5SUdPyjqCJVsN6qYapDpRWayCw5AP4gk+js0gWX8mQOhalR0j9xMIgHa5MCe7EGKaz4Z+moGqWVEgSvAuKS1IcTPUufC0NqrU8bVxwL2l8KGZ+e5UvIsc7s9DginxR4DAJhvvZutrL/YVdWeHNTXqk4K1L/psTDz1Jp6uI2QNyGbTBVpejJCQ09reeW2R1ZtebqGqmbrLdrre7Tn7AlK4J43l/G2b5Ry4rGjhrMZstKPkW/TbqtVwYuGyNler8XpDZvMiTK5W7br7S7wF9wXhrVvbETPg3indwT39snGjC87OrIk1KA4F/0lsQWoztc/a/H8ePMxrMqQYyODIDhr+kHz3GQe1A4RgbiXYJi7rwMHhViV48uekxkcP4mPDwR0hiSkwIYlFbvnBRN0jDjUXwbYq9j2Vu6lLnPsz6lhMSak3kOoofdvciaLIl0TYKZzvUqk1iq1+PUyV+PX9+2Wdh4Oj5+4T4erJYeM6Dp7uUJCniGlmyb215NmDsiD7TtcBcNldYDV4MPf4RWUxIfl0MqHe7e3/ArbYHi8=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
